# 2025-04-14 generate Gaussian grids

Adapted from this notebook from Spencer: https://github.com/ai2cm/explore/blob/master/spencer/2023-08-30-generate-gaussian-grids.ipynb.

In [5]:
!pip install "git+https://github.com/ai2cm/fv3net.git@68e4f378e933a9d90af9268a4f330cd6fc308e7d#egg=xtorch_harmonics&subdirectory=external/xtorch_harmonics"

  Using cached xtorch_harmonics-0.1.0-py3-none-any.whl
  Using cached torch_harmonics-0.5-py3-none-any.whl


In [6]:
import numpy as np
import xarray as xr
import xtorch_harmonics

In [7]:
def midpoints(bounds):
    return (bounds[:-1] + bounds[1:]) / 2


def compute_coordinates(n_lat, n_lon):
    lat_midpoints = xtorch_harmonics.xtorch_harmonics.compute_quadrature_latitudes(
        n_lat, "legendre-gauss"
    )
    lat_edges = np.concatenate(
        (np.array((-90,)), midpoints(lat_midpoints), np.array((90,)))
    )

    lon_edges = np.linspace(0, 360, n_lon + 1)
    lon_midpoints = midpoints(lon_edges)
    return lat_midpoints, lat_edges, lon_midpoints, lon_edges


def generate_grid(n_lat, n_lon):
    lat_midpoints, lat_edges, lon_midpoints, lon_edges = compute_coordinates(
        n_lat, n_lon
    )

    grid_lat = xr.DataArray(lat_edges, dims=["grid_y"], name="grid_lat")
    grid_latt = xr.DataArray(lat_midpoints, dims=["grid_yt"], name="grid_latt")
    grid_lon = xr.DataArray(lon_edges, dims=["grid_x"], name="grid_lon")
    grid_lont = xr.DataArray(lon_midpoints, dims=["grid_xt"], name="grid_lont")

    # Broadcast the coordinates against themselves to make a mesh grid
    # for the bounds and cell center coordinates.
    grid_lat, grid_lon = xr.broadcast(grid_lat, grid_lon)
    grid_latt, grid_lont = xr.broadcast(grid_latt, grid_lont)

    grid_lat.attrs["long_name"] = "Latitude cell edge"
    grid_lat.attrs["units"] = "degrees"
    grid_latt.attrs["long_name"] = "Latitude cell midpoint"
    grid_latt.attrs["units"] = "degrees"
    grid_lon.attrs["long_name"] = "Longitude cell edge"
    grid_lon.attrs["units"] = "degrees"
    grid_lont.attrs["long_name"] = "Longitude cell midpoint"
    grid_lont.attrs["units"] = "degrees"
    ds = xr.merge([grid_lat, grid_latt, grid_lon, grid_lont])
    return ds

In [8]:
four_degree = generate_grid(45, 90)
two_degree = generate_grid(90, 180)
quarter_degree = generate_grid(720, 1440)
half_degree = generate_grid(360, 720)

In [9]:
four_degree.to_netcdf("grids/gaussian_grid_45_by_90.nc")
two_degree.to_netcdf("grids/gaussian_grid_90_by_180.nc")
quarter_degree.to_netcdf("grids/gaussian_grid_720_by_1440.nc")
half_degree.to_netcdf("grids/gaussian_grid_360_by_720.nc")

Upload locally saved files to OSN Pod with:

1. Install rclone in terminal

    ```conda install conda-forge::rclone -y```
2. Setup rclone file if not already (https://leap-stc.github.io/guides/data_guides/external_workflow.html#authentication) 
3. Copy to OSN (https://leap-stc.github.io/guides/data_guides/external_workflow.html#moving-data)

   ```rclone copy grids ocean_emulator_write:emulators/sd5313/grids```
